# Threat Lateral Movement (BigID × Sentinel Data Lake)

**Question:** Which suspicious source IPs touched sensitive assets, and which downstream users also touched those same assets?
Joins BigID `RequestSourceIP` to `UserName` via shared `AssetID` to surface candidate lateral-movement paths.

In [ ]:
# === Setup: connect to the Microsoft Sentinel data lake ===
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

data_provider = MicrosoftSentinelProvider(spark)

WORKSPACE = "<YOUR_SENTINEL_WORKSPACE_NAME>"
TABLE = "BigIDDSPMCatalog_CL"

# Pull last 30 days of BigID catalog rows
df = data_provider.read_table(TABLE, WORKSPACE)
df = df.filter(F.col("TimeGenerated") > F.expr("current_timestamp() - INTERVAL 30 DAYS"))
df.printSchema()
print("Row count:", df.count())


## Build IP → Asset → User chains

In [ ]:
from pyspark.sql.functions import col

threat_rows = (
    df.filter(col("ThreatCategory").isNotNull() & (col("ThreatCategory") != ""))
      .select("RequestSourceIP", "AssetID", "ThreatCategory")
      .filter(col("RequestSourceIP").isNotNull())
)

asset_users = df.select("AssetID", "UserName").filter(col("UserName").isNotNull())

chains = (
    threat_rows.alias("t")
        .join(asset_users.alias("u"), "AssetID")
        .select("t.RequestSourceIP", "t.AssetID", "u.UserName", "t.ThreatCategory")
        .distinct()
)
chains.show(30, truncate=False)
print("Total chain rows:", chains.count())


## Flatten to edges for visualization

In [ ]:
# Two-hop graph: IP -> Asset, Asset -> User. Concatenate into a single edge list.
edges1 = chains.select(col("RequestSourceIP").alias("src"), col("AssetID").alias("dst"))
edges2 = chains.select(col("AssetID").alias("src"), col("UserName").alias("dst"))
result = edges1.union(edges2).distinct().limit(150)
result.show(20, truncate=False)


## Visualize

In [ ]:
# === Visualize as a graph ===
import matplotlib.pyplot as plt
import networkx as nx

pdf = result.toPandas()
print(f"Edges to draw: {len(pdf)}")
display(pdf.head(50))

G = nx.DiGraph()
for _, row in pdf.iterrows():
    src = str(row.iloc[0])
    dst = str(row.iloc[1])
    G.add_edge(src, dst)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 0].values],
    node_color="#1f77b4", node_size=900, alpha=0.85,
)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 1].values and n not in pdf.iloc[:, 0].values],
    node_color="#d62728", node_size=900, alpha=0.85,
)
nx.draw_networkx_edges(G, pos, arrows=True, edge_color="#888", alpha=0.6, width=1.2)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Threat Lateral Movement — Suspicious IP → Asset → User", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()
